In [0]:
# ── Load Silver Tables ─────────────────────────
spark.sql("USE CATALOG sap_catalog")

vbak_s = spark.table("sap_catalog.silver_schema.vbak_silver")
kna1_s = spark.table("sap_catalog.silver_schema.kna1_silver")
mara_s = spark.table("sap_catalog.silver_schema.mara_silver")
bkpf_s = spark.table("sap_catalog.silver_schema.bkpf_silver")
bseg_s = spark.table("sap_catalog.silver_schema.bseg_silver")

print("✅ Silver tables loaded!")
print(f"VBAK: {vbak_s.count():,} | KNA1: {kna1_s.count():,}")
print(f"BKPF: {bkpf_s.count():,} | BSEG: {bseg_s.count():,}")

In [0]:
from pyspark.sql.functions import (
    col, sum, count, avg, max,
    round, current_timestamp, lit
)

# ── VBAK Sales Summary (No Customer Join) ───────
sales_summary = (
    vbak_s
    .groupBy(
        "sales_order_number",
        "material_number",
        "plant",
        "division",
        "currency"
    )
    .agg(
        count("line_item_number").alias("total_line_items"),
        sum("order_quantity").alias("total_quantity"),
        sum("line_total").alias("total_revenue"),
        avg("net_price").alias("avg_unit_price")
    )
    .withColumn("total_revenue",
        round(col("total_revenue"), 2))
    .withColumn("avg_unit_price",
        round(col("avg_unit_price"), 2))
    .withColumn("_gold_ts", current_timestamp())
    .withColumn("_layer", lit("gold"))
    .orderBy(col("total_revenue").desc())
)

print(f"✅ Sales Summary: {sales_summary.count():,} orders")
display(sales_summary.limit(10))

In [0]:
# ── VBAK + MARA Join → Material Performance ────
material_perf = (
    vbak_s.alias("v")
    .join(mara_s.alias("m"),
          col("v.material_number") == col("m.material_number"),
          how="left")
    .groupBy(
        col("v.material_number"),
        col("m.material_type"),
        col("m.material_group"),
        col("v.division").alias("sales_division"),
        col("m.base_unit")
    )
    .agg(
        count("sales_order_number").alias("order_count"),
        sum("order_quantity").alias("total_quantity_sold"),
        sum("line_total").alias("total_revenue"),
        avg("net_price").alias("avg_price")
    )
    .withColumn("total_revenue",
        round(col("total_revenue"), 2))
    .withColumn("avg_price",
        round(col("avg_price"), 2))
    .withColumn("_gold_ts", current_timestamp())
    .withColumn("_layer", lit("gold"))
    .orderBy(col("total_revenue").desc())
)

print(f"✅ Material Performance: {material_perf.count():,} materials")
display(material_perf.limit(10))

In [0]:
# ── BKPF + BSEG → Financial Summary ──────────
financial_summary = (
    bkpf_s.alias("h")
    .join(bseg_s.alias("l"),
          on=["company_code","document_number","fiscal_year"],
          how="inner")
    .groupBy(
        col("h.company_code"),
        col("h.fiscal_year"),
        col("h.fiscal_period"),
        col("h.document_type"),
        col("l.account_type"),
        col("l.debit_credit")
    )
    .agg(
        count(col("h.document_number")).alias("transaction_count"),
        sum(col("l.amount_local")).alias("total_amount_local"),
        sum(col("l.amount_foreign")).alias("total_amount_foreign"),
        avg(col("l.amount_local")).alias("avg_amount")
    )
    .withColumn("total_amount_local",
        round(col("total_amount_local"), 2))
    .withColumn("total_amount_foreign",
        round(col("total_amount_foreign"), 2))
    .withColumn("avg_amount",
        round(col("avg_amount"), 2))
    .withColumn("_gold_ts", current_timestamp())
    .withColumn("_layer", lit("gold"))
    .orderBy("company_code", "fiscal_year", "fiscal_period")
)

print(f"✅ Financial Summary: {financial_summary.count():,} rows")
display(financial_summary.limit(10))

In [0]:
# ── Write Gold Tables ──────────────────────────
def write_gold(df, table_name):
    target = f"sap_catalog.gold_schema.{table_name}"
    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(target))
    print(f"✅ {target} — {df.count():,} rows")

write_gold(sales_summary,      "sales_order_summary")
write_gold(material_perf,      "material_performance")
write_gold(financial_summary,  "financial_summary")

print("\n" + "="*40)
print("🥇 GOLD LAYER COMPLETE!")
print("="*40)
display(spark.sql("SHOW TABLES IN sap_catalog.gold_schema"))

In [0]:
# ── Top Materials by Revenue ───────────────────
print("🏆 TOP 10 MATERIALS BY REVENUE:")
display(spark.sql("""
    SELECT 
        material_number,
        material_type,
        material_group,
        order_count,
        total_quantity_sold,
        total_revenue,
        avg_price
    FROM sap_catalog.gold_schema.material_performance
    ORDER BY total_revenue DESC
    LIMIT 10
"""))

# ── Financial Summary by Company ───────────────
print("💰 FINANCIAL SUMMARY BY COMPANY:")
display(spark.sql("""
    SELECT 
        company_code,
        fiscal_year,
        fiscal_period,
        document_type,
        transaction_count,
        total_amount_local,
        total_amount_foreign
    FROM sap_catalog.gold_schema.financial_summary
    ORDER BY company_code, fiscal_year, fiscal_period
    LIMIT 10
"""))